# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vasavijoshi/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Contract

- **Unit of analysis:** One row represents one content page for one day in the Search Console performance data.
- **Table:** `fact_content_daily_performance`, with `dim_content` used for page-level content attributes when needed.
- **Feature window:** February 2026.
- **Decision cutoff:** End of February 2026; features must be knowable by this cutoff.
- **Prediction/ranking:** Rank pages by their refresh/content opportunity using signals available before the decision cutoff.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

# Get the Hugging Face token securely from Colab Secrets
hf_token = userdata.get("HF_TOKEN")

# Create a DuckDB connection
con = duckdb.connect()

# Pass the secret through a DuckDB session variable
con.execute("SET VARIABLE hf_token = ?", [hf_token])

# Create the Hugging Face secret without exposing the token in the notebook
con.execute("""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN getvariable('hf_token')
);
""")

# Warehouse location
rel = "hf://datasets/FlyRank/internship-warehouse"

# Verify the grain for the February feature window
grain_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_grain_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
)
WHERE month = '2026-02';
""")

display(grain_check.df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_rows
0,7355108,7355108


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields: feature / label / context / excluded

### Feature
- `gsc_impressions`: search visibility available before the decision cutoff.
- `gsc_clicks`: search clicks available before the decision cutoff.
- `gsc_avg_position`: average search position available before the decision cutoff.
- Content metadata from `dim_content`: page-level attributes available before the decision cutoff.

### Label
- March search-performance outcome, derived from the subsequent label window, used to measure the page's future search opportunity.

### Context
- `client_hash_id`: identifies the pseudonymized client.
- `content_hash_id`: identifies the pseudonymized content item.
- `report_date`: identifies the observation date.

### Excluded
- Future-window search metrics: excluded because they would not be known at the decision moment.
- Label-derived fields: excluded from features because they would cause leakage.
- Raw pseudonymous IDs: excluded as model features because they identify entities rather than describe their search behavior.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
# Query 2: row count and date span for the February feature window

row_window_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
)
WHERE month = '2026-02';
""")
display(row_window_check.df())

,row_count,first_date,last_date
0,7355108,2026-02-01,2026-02-28


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 3: GSC availability in the February feature window

availability_check = con.sql(f"""
SELECT
    COUNT(*) AS gsc_available_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
)
WHERE month = '2026-02'
  AND gsc_data_available IS TRUE;
""")

display(availability_check.df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_available_rows
0,2621783


### Five-feature frame

For the February decision window, I will use these five features:

1. **February impressions** — available at the decision moment because Search Console impressions up to the end of February are already observed.
2. **February clicks** — available at the decision moment because Search Console clicks up to the end of February are already observed.
3. **February CTR** — available at the decision moment because it can be calculated from observed February clicks and impressions.
4. **February average position** — available at the decision moment because search position has already been measured in the February feature window.
5. **Number of measured GSC days** — available at the decision moment because GSC availability is known for each February observation day.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Five-feature frame: February 2026

feature_frame = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions_feb,
    SUM(gsc_clicks) AS clicks_feb,

    SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_feb,

    SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position_feb,

    COUNT(DISTINCT report_date) AS measured_gsc_days

FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
)

WHERE month = '2026-02'
  AND gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
""")

feature_df = feature_frame.df()

print(f"Feature rows: {len(feature_df):,}")
print(f"Feature columns: {len(feature_df.columns)}")

display(feature_df.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 153,559
Feature columns: 7


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,measured_gsc_days
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,0.000000,12.448161,28
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,0.008186,6.316508,28
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,0.000000,9.966926,28
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,0.001024,41.814739,28
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,0.002062,10.307216,28
5,client_e547b89c05043229,content_a2bd730a7cf68316,551.0,1.0,0.001815,6.154265,28
6,client_e547b89c05043229,content_cbe43d4b6ce2d320,291.0,0.0,0.000000,3.972509,28
7,client_e547b89c05043229,content_babd931911c9ee33,2680.0,31.0,0.011567,5.049254,28
8,client_e547b89c05043229,content_9c36ace83c73b5eb,416.0,1.0,0.002404,42.250000,28
9,client_e547b89c05043229,content_431784c057b25a5d,3641.0,6.0,0.001648,8.829992,28


### Leakage trap

To test for target leakage, I will deliberately add one feature derived from the future label window. This feature would not be available at the February decision cutoff.

If the feature makes the model score unrealistically well, that is evidence of leakage. I will then remove it and keep only features that were available at the decision moment.

In [6]:
# Deliberate leakage experiment
# Use a future-window signal as a deliberately leaked feature.

leakage_test = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions_feb,
    SUM(gsc_clicks) AS clicks_feb,

    -- Deliberately leaked: March clicks are from the future label window
    SUM(
        CASE
            WHEN month = '2026-03' THEN gsc_clicks
            ELSE 0
        END
    ) AS leaked_march_clicks

FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
)

WHERE month IN ('2026-02', '2026-03')
  AND gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
""")

leakage_df = leakage_test.df()

print(f"Rows in leakage experiment: {len(leakage_df):,}")
print("The March clicks feature is deliberately leaked because March is after the February decision cutoff.")

display(leakage_df.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in leakage experiment: 196,059
The March clicks feature is deliberately leaked because March is after the February decision cutoff.


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,leaked_march_clicks
0,client_e547b89c05043229,content_1eea820697c3b95a,614.0,0.0,0.0
1,client_e547b89c05043229,content_9abd8b303f805847,15269.0,10.0,4.0
2,client_e547b89c05043229,content_5f58c55cbfee172a,901.0,0.0,0.0
3,client_e547b89c05043229,content_6fe390ba3af1e456,7628.0,8.0,5.0
4,client_e547b89c05043229,content_3ad5d2160242b9ca,1974.0,3.0,1.0
5,client_e547b89c05043229,content_a2bd730a7cf68316,1102.0,2.0,1.0
6,client_e547b89c05043229,content_cbe43d4b6ce2d320,635.0,1.0,1.0
7,client_e547b89c05043229,content_babd931911c9ee33,7626.0,46.0,15.0
8,client_e547b89c05043229,content_9c36ace83c73b5eb,718.0,1.0,0.0
9,client_e547b89c05043229,content_431784c057b25a5d,5512.0,14.0,8.0


In [7]:
# Show the effect of the deliberate leakage

leakage_df["future_click_label"] = leakage_df["leaked_march_clicks"] > 0
leakage_df["leakage_prediction"] = leakage_df["leaked_march_clicks"] > 0

leakage_score = (
    leakage_df["future_click_label"]
    == leakage_df["leakage_prediction"]
).mean()

print(f"Leakage score (accuracy): {leakage_score:.4f}")
print("The near-perfect score is expected because the feature directly contains future label information.")

Leakage score (accuracy): 1.0000
The near-perfect score is expected because the feature directly contains future label information.


### Leakage removed

The deliberately leaked March feature is removed from the final feature set.

The final ranking features are restricted to February information only: impressions, clicks, CTR, impression-weighted average position, and measured GSC days. These were all available by the end-of-February decision cutoff.


## 4. Data limits

One limitation is that GSC availability is not complete for every page and every day. The feature frame therefore uses only observations where `gsc_data_available IS TRUE`, so pages with limited GSC coverage may have less reliable feature values.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.